In [1]:
# Installation RecBole (une seule fois dans le terminal conda llmkger)
# pip install recbole
# pip install ray  (pour hyperparameter tuning, optionnel)

import torch
from importlib.metadata import version

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
try:
    print(f'RecBole : {version("recbole")}')
except:
    print('RecBole : NON installé → pip install recbole')


PyTorch : 2.6.0+cu124
GPU     : True
RecBole : 1.2.1


## Étape 1 — Préparer les données au format RecBole

RecBole exige un format spécifique :
- `dataset.inter` : interactions user-item (user_id:token, item_id:token)
- Fichiers séparés par tabulation avec types de colonnes


In [2]:
import os
inter_path = r'D:\Memoire\LLMKGER\recbole_data\llmkger\llmkger.inter'
print(f'Fichier .inter existe : {os.path.exists(inter_path)}')
if os.path.exists(inter_path):
    size = os.path.getsize(inter_path) / 1024 / 1024
    print(f'Taille : {size:.1f} MB')

Fichier .inter existe : True
Taille : 115.6 MB


In [2]:
import pandas as pd
import os

KG_DIR     = r'D:\Memoire\LLMKGER\kg_final'
RECBOLE_DIR = r'D:\Memoire\LLMKGER\recbole_data\llmkger'
os.makedirs(RECBOLE_DIR, exist_ok=True)

# Charger les interactions
train = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
valid = pd.read_csv(os.path.join(KG_DIR, 'interactions_valid.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
test  = pd.read_csv(os.path.join(KG_DIR, 'interactions_test.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])

# Combiner tout (RecBole fera son propre split)
all_inter = pd.concat([train, valid, test], ignore_index=True)
all_inter = all_inter[['user_id', 'item_id']].copy()
all_inter['rating'] = 1  # interaction implicite

print(f'Interactions totales : {len(all_inter):,}')
print(f'Users  uniques : {all_inter["user_id"].nunique():,}')
print(f'Items  uniques : {all_inter["item_id"].nunique():,}')

# Écrire au format RecBole .inter
inter_path = os.path.join(RECBOLE_DIR, 'llmkger.inter')
with open(inter_path, 'w', encoding='utf-8') as f:
    f.write('user_id:token\titem_id:token\trating:float\n')  # header typé
    for _, row in all_inter.iterrows():
        f.write(f"{row['user_id']}\t{row['item_id']}\t{row['rating']}\n")

print(f'\n✅ Fichier .inter créé → {inter_path}')


Interactions totales : 5,258,360
Users  uniques : 106,735
Items  uniques : 2,822

✅ Fichier .inter créé → D:\Memoire\LLMKGER\recbole_data\llmkger\llmkger.inter


## Étape 2 — Configuration commune RecBole


In [3]:
# Configuration partagée par tous les modèles
import os

RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data'
RESULTS_REC  = r'D:\Memoire\LLMKGER\reco_results'
os.makedirs(RESULTS_REC, exist_ok=True)

base_config = {
    'data_path'        : RECBOLE_ROOT,
    'dataset'          : 'llmkger',
    'USER_ID_FIELD'    : 'user_id',
    'ITEM_ID_FIELD'    : 'item_id',
    'RATING_FIELD'     : 'rating',
    'load_col'         : {'inter': ['user_id', 'item_id', 'rating']},
    # Split LOO comme Yasmine
    'eval_args': {
        'split'     : {'LS': 'valid_and_test'},  # Leave-one-out split
        'order'     : 'RO',                       # Random order
        'group_by'  : 'user',
        'mode'      : 'full',
    },
    # Métriques (mêmes que Yasmine)
    'metrics'    : ['Recall', 'NDCG', 'Precision', 'Hit', 'MRR'],
    'topk'       : [5, 10],
    'valid_metric': 'NDCG@10',
    # Entraînement
    'epochs'         : 50,
    'train_batch_size': 4096,
    'eval_batch_size' : 4096,
    'learning_rate'  : 0.001,
    'embedding_size' : 64,
    'stopping_step'  : 5,   # early stopping patience
    'device'         : 'cuda',
    'seed'           : 42,
}

print('Config commune définie ✅')
print(f'Métriques : {base_config["metrics"]} @ {base_config["topk"]}')


Config commune définie ✅
Métriques : ['Recall', 'NDCG', 'Precision', 'Hit', 'MRR'] @ [5, 10]


In [4]:
# Bypass ray complètement — on utilise l'API RecBole sans run_recbole
import os, json
import torch
from logging import getLogger

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model, get_trainer

def train_model(model_name, config_dict):
    """Entraîne un modèle RecBole sans passer par run_recbole (évite ray)."""
    config = Config(model=model_name, dataset='llmkger', config_dict=config_dict)
    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()

    # Dataset
    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)

    # Modèle
    model = get_model(config['model'])(config, train_data._dataset).to(config['device'])

    # Trainer
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    # Entraînement
    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=True, show_progress=True
    )

    # Évaluation test
    test_result = trainer.evaluate(test_data)

    return test_result

print('Fonction train_model définie ✅ (sans ray)')

Fonction train_model définie ✅ (sans ray)


## Modèle 1 — BPRMF (Baseline)


In [5]:
import pandas as pd
import os

KG_DIR = r'D:\Memoire\LLMKGER\kg_final'
train = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])

print(f'Interactions train : {len(train):,}')
print(f'Users  : {train["user_id"].nunique():,}')
print(f'Items  : {train["item_id"].nunique():,}')
counts = train.groupby('user_id').size()
print(f'Moyenne interactions/user : {counts.mean():.1f}')
print(f'Max : {counts.max()}')

Interactions train : 5,044,890
Users  : 106,735
Items  : 2,822
Moyenne interactions/user : 47.3
Max : 2589


In [ ]:
import json, os

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'
os.makedirs(RESULTS_REC, exist_ok=True)

light_config = {**base_config,
    'train_batch_size': 2048,
    'eval_batch_size' : 2048,
    'epochs'          : 30,
}

print('=== Entraînement BPRMF ===')
test_result_bpr = train_model('BPR', light_config)

print('\nRésultats BPRMF :')
for k, v in test_result_bpr.items():
    print(f'  {k} : {v:.4f}')

with open(os.path.join(RESULTS_REC, 'BPRMF_results.json'), 'w') as f:
    json.dump({k: float(v) for k, v in test_result_bpr.items()}, f, indent=2)
print('\nBPRMF sauvegardé ✅')

=== Entraînement BPRMF ===


c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

## Modèle 2 — NGCF (Baseline)


In [ ]:
from recbole.quick_start import run_recbole
import json, os

print('=== Entraînement NGCF ===')
result_ngcf = run_recbole(
    model='NGCF',
    dataset='llmkger',
    config_dict=base_config
)

print('\nRésultats NGCF :')
print(result_ngcf['test_result'])

with open(os.path.join(RESULTS_REC, 'NGCF_results.json'), 'w') as f:
    json.dump({k: float(v) for k,v in result_ngcf['test_result'].items()}, f, indent=2)
print('NGCF sauvegardé ✅')


## Modèle 3 — LightGCN (Baseline)


In [ ]:
from recbole.quick_start import run_recbole
import json, os

print('=== Entraînement LightGCN ===')
result_lgcn = run_recbole(
    model='LightGCN',
    dataset='llmkger',
    config_dict={**base_config, 'n_layers': 3}
)

print('\nRésultats LightGCN :')
print(result_lgcn['test_result'])

with open(os.path.join(RESULTS_REC, 'LightGCN_results.json'), 'w') as f:
    json.dump({k: float(v) for k,v in result_lgcn['test_result'].items()}, f, indent=2)
print('LightGCN sauvegardé ✅')


## Modèle 4 — KGAT (= E-KGAT base)

⚠️ KGAT nécessite un fichier `.kg` (knowledge graph) en plus du `.inter`


In [ ]:
# Préparer le fichier .kg et .link pour KGAT
import pandas as pd
import os

# Charger le KG sémantique
df_kg = pd.read_csv(os.path.join(KG_DIR, 'kg_train.tsv'),
                    sep='\t', header=0, names=['head','relation','tail'])
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()

# Fichier .kg : head_id, relation_id, tail_id
kg_path = os.path.join(RECBOLE_DIR, 'llmkger.kg')
with open(kg_path, 'w', encoding='utf-8') as f:
    f.write('head_id:token\trelation_id:token\ttail_id:token\n')
    for _, row in df_kg_sem.iterrows():
        f.write(f"{row['head']}\t{row['relation']}\t{row['tail']}\n")
print(f'Fichier .kg créé : {len(df_kg_sem):,} triplets')

# Fichier .link : item_id → entity_id (les cours sont les items ET des entités du KG)
items = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                    sep='\t', header=0, names=['user_id','relation','item_id'])
unique_items = items['item_id'].unique()

link_path = os.path.join(RECBOLE_DIR, 'llmkger.link')
with open(link_path, 'w', encoding='utf-8') as f:
    f.write('item_id:token\tentity_id:token\n')
    for item in unique_items:
        f.write(f"{item}\t{item}\n")  # cours = entité du même nom
print(f'Fichier .link créé : {len(unique_items):,} liens')


In [ ]:
from recbole.quick_start import run_recbole
import json, os

# Config KGAT avec chargement du KG
kgat_config = {**base_config,
    'load_col': {
        'inter': ['user_id', 'item_id', 'rating'],
        'kg'   : ['head_id', 'relation_id', 'tail_id'],
        'link' : ['item_id', 'entity_id'],
    },
}

print('=== Entraînement KGAT ===')
result_kgat = run_recbole(
    model='KGAT',
    dataset='llmkger',
    config_dict=kgat_config
)

print('\nRésultats KGAT :')
print(result_kgat['test_result'])

with open(os.path.join(RESULTS_REC, 'KGAT_results.json'), 'w') as f:
    json.dump({k: float(v) for k,v in result_kgat['test_result'].items()}, f, indent=2)
print('KGAT sauvegardé ✅')


## Étape 3 — Comparaison des 4 modèles


In [ ]:
import json, os
import pandas as pd

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results'

comparison = {}
for model in ['BPRMF', 'NGCF', 'LightGCN', 'KGAT']:
    path = os.path.join(RESULTS_REC, f'{model}_results.json')
    if os.path.exists(path):
        with open(path) as f:
            comparison[model] = json.load(f)
        print(f'{model} chargé ✅')
    else:
        print(f'{model} — non trouvé')

df_comp = pd.DataFrame(comparison).T
print('\n' + '='*70)
print('  Comparaison des modèles de recommandation')
print('='*70)
print(df_comp.to_string())

# Meilleur selon NDCG@10
if 'ndcg@10' in df_comp.columns:
    best = df_comp['ndcg@10'].idxmax()
    print(f'\n🏆 Meilleur modèle : {best}')

df_comp.to_csv(os.path.join(RESULTS_REC, 'reco_comparison.csv'))
print('Tableau sauvegardé ✅')
